# RAG Pipeline — MongoDB Atlas Vector Store
### Support Chatbot · LangChain + LangGraph + OpenAI
Step-by-step notebook to build, test, and tune the retrieval pipeline before wiring it into the Streamlit app.


In [ ]:
# Install all required packages
# Run once; restart kernel after if needed
%pip install -q langchain langchain-openai langchain-community langgraph \
               pymongo langchain-mongodb python-dotenv pypdf \
               langchain-text-splitters tiktoken

## 1. Imports & Configuration

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_mongodb import MongoDBAtlasVectorSearch
from langchain_core.messages import SystemMessage, HumanMessage, BaseMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

from pymongo import MongoClient
from typing import TypedDict, Annotated
from dotenv import load_dotenv
import os, textwrap, pprint

load_dotenv()

## 2. Configuration — Set Once, Use Everywhere

In [ ]:
# ── LLM & Embedding models ────────────────────────────────────────────
LLM_MODEL       = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"  # 1536-dim, cheap & accurate

# ── MongoDB Atlas ──────────────────────────────────────────────────────
MONGO_URI          = os.getenv("MONGO_URI")          # Atlas connection string
DB_NAME            = "support_bot"
COLLECTION_NAME    = "knowledge_base"
VECTOR_INDEX_NAME  = "vector_index"                  # Atlas Search index name

# ── Chunking strategy ──────────────────────────────────────────────────
CHUNK_SIZE    = 700    # tokens — tunable (see Section 6)
CHUNK_OVERLAP = 150    # ~20% overlap keeps cross-chunk context

# ── Retrieval ──────────────────────────────────────────────────────────
RETRIEVER_K       = 5   # candidate docs to retrieve
RETRIEVER_TYPE    = "mmr"  # "similarity" | "mmr"
MMR_FETCH_K       = 20  # MMR candidate pool before re-ranking
MMR_LAMBDA        = 0.7 # 1.0 = pure similarity, 0.0 = pure diversity

print("✅ Config loaded")
print(f"   LLM: {LLM_MODEL}  |  Embeddings: {EMBEDDING_MODEL}")
print(f"   DB:  {DB_NAME}.{COLLECTION_NAME}  |  Index: {VECTOR_INDEX_NAME}")

## 3. Initialize LLM & Embeddings

In [ ]:
llm = ChatOpenAI(model=LLM_MODEL, temperature=0.3)  # lower temp = more factual
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

# Quick sanity check
test_embed = embeddings.embed_query("test")
print(f"✅ Embedding dim: {len(test_embed)}")
print(f"✅ LLM: {llm.model_name}")

## 4. Load Documents
Supports single PDF or a folder of PDFs.

In [ ]:
# ── Option A: Single PDF ──────────────────────────────────────────────
PDF_PATH = "your_document.pdf"   # ← change to your file path

loader = PyPDFLoader(PDF_PATH)
raw_docs = loader.load()

# ── Option B: Load entire folder (uncomment to use) ───────────────────
# loader = DirectoryLoader("./docs/", glob="**/*.pdf", loader_cls=PyPDFLoader)
# raw_docs = loader.load()

print(f"✅ Loaded {len(raw_docs)} page(s)")
print(f"   First 300 chars: {raw_docs[0].page_content[:300]!r}")

## 5. Chunking
`RecursiveCharacterTextSplitter` tries larger separators first (paragraphs → sentences → words),
so chunks respect natural text boundaries.

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size    = CHUNK_SIZE,
    chunk_overlap = CHUNK_OVERLAP,
    separators    = ["\n\n", "\n", ". ", " ", ""],  # priority order
    length_function = len,
)

chunks = text_splitter.split_documents(raw_docs)

print(f"✅ {len(raw_docs)} pages  →  {len(chunks)} chunks")
print(f"   Avg chunk length: {sum(len(c.page_content) for c in chunks)//len(chunks)} chars")
print(f"\n── Sample chunk ──────────────────────────────")
print(textwrap.fill(chunks[10].page_content, 100))
print(f"\n── Metadata ───────────────────────────────────")
pprint.pprint(chunks[10].metadata)

## 6. Connect to MongoDB Atlas
Make sure your Atlas cluster has:
1. A database `support_bot` and collection `knowledge_base` created.
2. An **Atlas Vector Search index** named `vector_index` on the `embedding` field
   with `numDimensions: 1536` (matches `text-embedding-3-small`).
   
   *Index JSON to paste in Atlas UI:*
   ```json
   {
     "fields": [{
       "type": "vector",
       "path": "embedding",
       "numDimensions": 1536,
       "similarity": "cosine"
     }]
   }
   ```


In [ ]:
client     = MongoClient(MONGO_URI)
db         = client[DB_NAME]
collection = db[COLLECTION_NAME]

# Verify connection
client.admin.command("ping")
print(f"✅ Connected to MongoDB Atlas")
print(f"   Existing docs in collection: {collection.count_documents({})}")

## 7. Embed & Ingest into MongoDB
⚠️ **Run this cell only once** (or when you update your documents).  
Re-running without clearing the collection will create duplicate vectors.

In [ ]:
# ── Optional: clear existing vectors before re-ingesting ─────────────
# collection.delete_many({})
# print("🗑️  Collection cleared")

vector_store = MongoDBAtlasVectorSearch.from_documents(
    documents       = chunks,
    embedding       = embeddings,
    collection      = collection,
    index_name      = VECTOR_INDEX_NAME,
)

print(f"✅ Ingested {len(chunks)} chunks into MongoDB Atlas")
print(f"   Total docs in collection: {collection.count_documents({})}")

## 8. Load Existing Vector Store (use after first ingest)

In [ ]:
# Use this cell on subsequent runs — skip Section 7 after first ingest
vector_store = MongoDBAtlasVectorSearch(
    collection     = collection,
    embedding      = embeddings,
    index_name     = VECTOR_INDEX_NAME,
    text_key       = "text",         # field storing chunk text
    embedding_key  = "embedding",    # field storing the vector
)

print("✅ Vector store loaded from MongoDB Atlas")

## 9. Configure Retriever
**MMR (Maximal Marginal Relevance)** is recommended over plain similarity:  
it balances *relevance* and *diversity* so you don't get 5 near-identical chunks.

In [ ]:
# ── MMR retriever (recommended) ────────────────────────────────────
retriever = vector_store.as_retriever(
    search_type   = RETRIEVER_TYPE,
    search_kwargs = {
        "k":        RETRIEVER_K,
        "fetch_k":  MMR_FETCH_K,
        "lambda_mult": MMR_LAMBDA,
    }
)

# ── Quick retrieval test ─────────────────────────────────────────────
test_query  = "How do I create an invoice?"   # ← change to suit your doc
test_result = retriever.invoke(test_query)

print(f"✅ Retrieved {len(test_result)} chunks for: '{test_query}'")
print()
for i, doc in enumerate(test_result, 1):
    print(f"── Chunk {i} (page {doc.metadata.get('page', '?')}) ───────────────")
    print(textwrap.fill(doc.page_content[:300], 100))
    print()

## 10. System Prompt
The prompt is the single biggest lever for accuracy.  
Key principles applied here:
- Strict grounding — answer ONLY from context
- Explicit fallback — what to say when the answer isn't there
- Structured tone — professional support assistant


In [ ]:
SYSTEM_PROMPT = """You are an expert support assistant for [Your Organization Name].
Your job is to help the support team by answering questions accurately and concisely.

STRICT RULES:
1. Answer ONLY using the information in the context provided below.
2. If the context does not contain enough information to answer, respond with:
   "I don't have enough information to answer this. Please escalate to a senior agent or refer to [resource]."
3. Do NOT use any outside knowledge or make assumptions beyond what is stated.
4. If the question is ambiguous, ask a single clarifying question before answering.
5. Keep answers concise — use bullet points for step-by-step instructions.
6. Always cite the source page number if available, e.g., (Source: Page 12).

Context:
{context}
"""

print("✅ System prompt defined")
print(f"   Length: {len(SYSTEM_PROMPT)} chars")

## 11. Build the RAG Chain

In [ ]:
def format_docs(docs):
    """Concatenate retrieved chunks with page-number annotation."""
    parts = []
    for doc in docs:
        page = doc.metadata.get("page", "?")
        parts.append(f"[Page {page}]\n{doc.page_content}")
    return "\n\n---\n\n".join(parts)


prompt_template = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human",  "{question}"),
])

# Vanilla RAG chain (no memory — for testing retrieval quality)
rag_chain = (
    {
        "context":  retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt_template
    | llm
    | StrOutputParser()
)

print("✅ RAG chain built")

## 12. Test the RAG Chain

In [ ]:
def ask(question: str):
    """Run a question through the RAG chain and pretty-print the answer."""
    print(f"❓ {question}")
    print("─" * 60)
    answer = rag_chain.invoke(question)
    print(answer)
    print()
    return answer

# ── Add your test questions here ────────────────────────────────────
ask("How do I create an invoice?")
ask("What is the process for handling a refund?")
ask("Who should I contact for billing issues?")

## 13. Retrieval Diagnostics
Run this to see *exactly* what context the LLM received — 
the fastest way to debug wrong or missing answers.

In [ ]:
def diagnose(question: str):
    """Show retrieved chunks + similarity scores for a question."""
    print(f"🔍 Diagnosing: '{question}'")
    print("=" * 70)
    
    # Raw similarity search with scores
    results = vector_store.similarity_search_with_score(question, k=RETRIEVER_K)
    
    for i, (doc, score) in enumerate(results, 1):
        page = doc.metadata.get("page", "?")
        print(f"\n── Chunk {i}  |  Score: {score:.4f}  |  Page: {page} ──")
        print(textwrap.fill(doc.page_content[:400], 100))
    
    print("\n" + "=" * 70)

diagnose("How do I create an invoice?")

## 14. Tuning Guide — Improving Accuracy

### A. Chunking
| Problem | Fix |
|---|---|
| Answer spans two chunks | Increase `CHUNK_OVERLAP` (try 200–250) |
| Chunks too noisy / off-topic | Decrease `CHUNK_SIZE` (try 400–500) |
| Tables/lists get split | Add `"\n"` as a separator, or use `MarkdownHeaderTextSplitter` for structured docs |

### B. Retrieval
| Problem | Fix |
|---|---|
| Repetitive chunks retrieved | Use MMR, raise `MMR_FETCH_K`, lower `MMR_LAMBDA` |
| Correct chunk ranked too low | Increase `k` (try 6–8) |
| Slow retrieval | Decrease `MMR_FETCH_K` |

### C. Embedding Model
- `text-embedding-3-small` (1536-dim) — good balance of cost and accuracy ✅  
- `text-embedding-3-large` (3072-dim) — higher accuracy, ~2× cost  
- Test with your domain: embed 10 queries, manually check top-3 results

### D. LLM Temperature
- `0.0–0.3` → factual, consistent (recommended for support bots)  
- `0.7+` → more creative, less reliable for factual Q&A

### E. Prompt Engineering
- Add **few-shot examples** inside the system prompt for edge cases  
- Add **negative examples** ("If asked about X, do not...")  
- Include **output format instructions** (bullets, numbered steps, tables)

### F. Re-ranking (Advanced)
After retrieval, add a cross-encoder re-ranker (e.g. `FlashrankRerank` or Cohere Rerank)
to re-score the `k` retrieved chunks before passing to the LLM. Often gives the biggest
accuracy jump for domain-specific docs.


## 15. Experiment — Compare Chunk Sizes

In [ ]:
# Run this to compare how different chunk sizes affect retrieved context quality
# Helps you pick the best CHUNK_SIZE for your document

TEST_QUERY = "How do I create an invoice?"  # ← use a real question from your domain

for chunk_size in [400, 600, 800]:
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=100)
    exp_chunks = splitter.split_documents(raw_docs)
    
    exp_store = MongoDBAtlasVectorSearch.from_documents(
        documents  = exp_chunks[:50],   # use first 50 chunks only for speed
        embedding  = embeddings,
        collection = db["experiment_chunks"],
        index_name = VECTOR_INDEX_NAME,
    )
    exp_ret = exp_store.as_retriever(search_type="mmr", search_kwargs={"k": 3})
    results = exp_ret.invoke(TEST_QUERY)
    
    print(f"\n{'='*60}")
    print(f"Chunk size: {chunk_size}  |  Total chunks: {len(exp_chunks)}")
    print(f"Top result ({len(results[0].page_content)} chars):")
    print(textwrap.fill(results[0].page_content[:300], 90))

# Clean up experiment collection
db["experiment_chunks"].delete_many({})
print("\n✅ Experiment done — temp collection cleared")

## 16. Next Steps (after notebook sign-off)

Once you're happy with the retrieval quality here, the integration path into the app is:

1. **`backend.py`** — add a `retriever_node` before `chat_node` in the LangGraph graph  
   that fetches context and injects it as a `SystemMessage`
2. **Add `MemorySaver`** checkpointer for real session memory
3. **`ui.py`** — add a PDF uploader in the sidebar so the support team can update the knowledge base
4. **MongoDB index** — add a metadata filter field (e.g. `department`) for multi-tenant setups
